# Nemotron Post-Training Lab — Google Colab Quickstart

Run **SFT → GRPO → eval** on a free **T4 GPU** (no local GPU needed).

**Before you start:** Runtime → Change runtime type → **T4 GPU**

**Setup options**
- **Option A (recommended):** Push this repo to GitHub and set `REPO_URL` below
- **Option B:** Upload the project folder to Google Drive and set `DRIVE_PROJECT_PATH`

In [ ]:
# --- CONFIG ---
REPO_URL = ""  # e.g. "https://github.com/sahmed-46/nemotron-posttraining-lab.git"
DRIVE_PROJECT_PATH = ""  # e.g. "/content/drive/MyDrive/nemotron-posttraining-lab"

RUN_GRPO = True   # set False if you only want SFT (faster)
SAVE_TO_DRIVE = True

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path("/content/nemotron-posttraining-lab")

if REPO_URL:
    if PROJECT_DIR.exists():
        !rm -rf {PROJECT_DIR}
    !git clone {REPO_URL} {PROJECT_DIR}
elif DRIVE_PROJECT_PATH:
    from google.colab import drive
    drive.mount("/content/drive")
    src = Path(DRIVE_PROJECT_PATH)
    assert src.exists(), f"Not found: {src}"
    if PROJECT_DIR.exists():
        !rm -rf {PROJECT_DIR}
    !cp -r "{src}" {PROJECT_DIR}
else:
    raise ValueError("Set REPO_URL or DRIVE_PROJECT_PATH in the config cell.")

%cd {PROJECT_DIR}
!ls -la

In [ ]:
# Install PyTorch stack + TRL (Colab usually has torch; reinstall if needed)
!pip install -q -r requirements.txt
!pip install -q accelerate peft bitsandbytes transformers trl datasets wandb

In [ ]:
# Prepare GSM8K subsets for SFT / DPO / GRPO
!python scripts/prepare_datasets.py --config configs/colab.yaml

In [ ]:
# SFT warm-start (required before GRPO)
!python scripts/train_sft.py --config configs/colab.yaml --method-config configs/sft.yaml

In [ ]:
SFT_CKPT = "outputs/checkpoints/sft/final"

if RUN_GRPO:
    !python scripts/train_grpo.py --config configs/colab.yaml --method-config configs/grpo.yaml --model-path {SFT_CKPT}
    EVAL_CKPT = "outputs/checkpoints/grpo/final"
else:
    EVAL_CKPT = SFT_CKPT

print("Evaluating:", EVAL_CKPT)

In [ ]:
!python scripts/run_eval.py --config configs/colab.yaml --model-path {EVAL_CKPT} --output outputs/eval/colab_report.json

import json
from pathlib import Path
report = json.loads(Path("outputs/eval/colab_report.json").read_text())
print(f"Accuracy: {report['accuracy']:.2%} on {report['total']} examples")

In [ ]:
# Optional: copy results to Google Drive
if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    from datetime import datetime

    drive.mount("/content/drive", force_remount=True)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    dest = Path(f"/content/drive/MyDrive/nemotron-posttraining-lab_runs/{stamp}")
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copytree("outputs", dest / "outputs", dirs_exist_ok=True)
    print("Saved to:", dest)